# Pilot evaluation of the complaint lexicon: draw the sample to label

Runs on a free Colab **CPU** runtime in about 5-10 minutes; no GPU and no token are needed.
It follows the locked protocol in `docs/annotation/pilot_protocol.md` and ends by
downloading `pilot_sample.zip`.

1. **Raw data.** `scripts/fetch_booking_515k.py` downloads the Booking 515K CSV and
   refuses it unless its size and SHA-256 match the Kaggle file.
2. **Sample.** `scripts/annotation_sample.py` draws the texts to label from the negative
   fields of the 863 compared hotels, February-July 2016 and February-July 2017:
   - 100 random texts per period;
   - 50 texts per period in which the lexicon named one of the five topics;
   - 80 of the 300 texts also go to the second annotator.

**After the download:**
- **`sheet_A.csv` is for you: 300 texts.** In every topic cell, write `1`, `0` or
  `unsure`, and write `yes` in `done`. A blank cell sends the whole sheet back.
- **`sheet_B.csv` goes to the second annotator**, together with
  `complaint_topics_guideline.md`. Do not discuss the texts before both sheets are handed in.
- **Do not open `key.csv`.** It says which texts the lexicon matched.
- **Send back** the two filled sheets and `key.csv`, as they are.

In [ ]:
!git clone --quiet --branch claude/funny-rubin-31xgtb https://github.com/krimits/hotel-review-nlp.git
%cd hotel-review-nlp
!git log --oneline -1
!pip install --quiet --no-deps -e .

In [ ]:
import subprocess


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


run("python scripts/fetch_booking_515k.py")

## Draw the sample (seed 0, as the protocol fixes)

In [ ]:
run("python scripts/annotation_sample.py --workers 2 --expect-hotels 863 --out runs/annotation")

In [ ]:
import json

manifest = json.load(open("runs/annotation/manifest.json"))
print("protocol", manifest["protocol_sha256"][:12], "| code", manifest["code_commit"][:7],
      "| hotels", manifest["hotels_compared"], "| texts", manifest["items"],
      "| second annotator", manifest["second_annotator_items"])
for name, numbers in manifest["periods"].items():
    print(name, "texts", numbers["population"], "| M drawn", numbers["M"],
          "| shortfall", numbers["M_shortfall"], "| set aside by the filter", numbers["excluded_by_filter"])

## Download the sheets, the key and the guideline

In [ ]:
import zipfile

from google.colab import files

paths = ["runs/annotation/sheet_A.csv", "runs/annotation/sheet_B.csv", "runs/annotation/key.csv",
         "runs/annotation/manifest.json", "docs/annotation/complaint_topics_guideline.md"]
with zipfile.ZipFile("pilot_sample.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path, arcname=path.rsplit("/", 1)[-1])
files.download("pilot_sample.zip")